# 阶段 07 · 保存 adapter (Colab)

理解 adapter 内容, 并把它持久化保存(Colab 磁盘是临时的, 不存就丢)。

- 方式 1: 复制到 Google Drive (最简单)
- 方式 2: 推送到 HuggingFace Hub (可分享/版本管理, 可选)

In [ ]:
# [Cell 1] 进入项目 (从 GitHub clone 的目录, 或 Drive)
%cd /content/Lora
# 查看 adapter 内容、大小、配置
!python src/inspect_adapter.py

In [ ]:
# [Cell 2] (可选) 删除中间 checkpoint, 只留最终 adapter, 省空间
!rm -rf outputs/lora_adapter/checkpoint-*
!python src/inspect_adapter.py

In [ ]:
# [Cell 3] 方式1: 持久化到 Google Drive (带版本号命名)
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
SAVE_NAME = "identity_r16_v1"   # 版本化命名, 方便回溯
dst = f"/content/drive/MyDrive/lora_adapters/{SAVE_NAME}"
os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copytree("outputs/lora_adapter", dst, dirs_exist_ok=True)
print("已保存到:", dst)

In [ ]:
# [Cell 4] 方式2 (可选): 推送到 HuggingFace Hub
# 需要先在 https://huggingface.co/settings/tokens 创建 write 权限 token
from huggingface_hub import login
login()  # 运行后粘贴你的 HF token

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="outputs/lora_adapter",
    max_seq_length=512,
    load_in_4bit=True,
    dtype=None,
)
# 把 <用户名> 换成你的 HF 用户名
repo_id = "<你的HF用户名>/qwen3-1.7b-xiaojing-lora"
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)
print("已推送到 HuggingFace Hub:", repo_id)